(z3-knight)=
# 骑士巡游 ― 寻找把每个格子恰好访问一次的路线

## 本章要点

让国际象棋的骑士，在 8×8 的棋盘上找出把**每个格子恰好访问一次**的路线。最后还要回到起点，形成闭合路线。

- 给每个格子分配一个表示「第几个访问」的整数
- 用 `PbEq()` 写出「相邻的两个编号之间是骑士的走法」
- 把规则原样变成约束条件，Z3 几秒内就能找到路线

骑士巡游问题在 [用 OrTools 求解骑士巡游问题](ortools-knight) 中也有讲解。

## 思路

给每个格子分配 `0` 到 `63` 的整数，让**这个数表示「第几个访问」**。

- 所有格子的值都在 0〜63 之间，并且互不相同（`Distinct`）
- 值为 `c` 的格子周围，一定有一个值为 `c+1` 的格子
- 值为 `c` 的格子周围，一定有一个值为 `c-1` 的格子
- 最后一个（第 63 个）格子的周围，一定有第 0 个格子（为了形成闭合路线）

这里的「周围」只算骑士走法（L 字形 8 个方向）能到达的格子。

这个约束只能在骑士能走到的范围内满足，所以路线自然就被定下来了。

In [1]:
from z3 import *
from itertools import product

# 骑士的 8 个走法方向
KNIGHT_MOVES = [(1, 2), (2, 1), (-1, 2), (2, -1), (-2, 1), (1, -2), (-1, -2), (-2, -1)]

## 代码

给棋盘的每个格子创建一个整数变量，然后添加约束条件。

In [2]:
w, h = 8, 8
N = w * h

cells = {}
solver = Solver()

for i, j in product(range(h), range(w)):
    c = cells[i, j] = Int(f'c_{i}_{j}')
    solver.add(c >= 0, c <= N - 1)

# 访问顺序互不相同
solver.add(Distinct(list(cells.values())))

for i, j in product(range(h), range(w)):
    c = cells[i, j]
    neighbours = [cells[i + di, j + dj] for di, dj in KNIGHT_MOVES
                  if (i + di, j + dj) in cells]

    next_is = [(x == c + 1, 1) for x in neighbours]   # 周围是否有 c+1
    prev_is = [(x == c - 1, 1) for x in neighbours]   # 周围是否有 c-1
    start_is = [(x == 0, 1) for x in neighbours]      # 周围是否有 0

    # 除最后一个格子（63）外，周围必须有一个 c+1
    solver.add(Implies(c < N - 1, PbEq(next_is, 1)))
    # 除第一个格子（0）外，周围必须有一个 c-1
    solver.add(Implies(c > 0, PbEq(prev_is, 1)))
    # 最后一个格子（63）的周围必须有 0（闭合路线）
    solver.add(Implies(c == N - 1, PbEq(start_is, 1)))

%time solver.check()
m = solver.model()

for i in range(h):
    for j in range(w):
        print(f'{m[cells[i, j]].as_long():2d}', end=' ')
    print()

CPU times: total: 4 s
Wall time: 5.02 s
63 60 45 10 47 58 23 20 
44 11 62 59 24 21 48 57 
61  0  9 46 27 56 19 22 
12 43 36  3 54 25 28 49 
37  8  1 26 35  4 55 18 
42 13 40 53  2 31 50 29 
 7 38 15 34  5 52 17 32 
14 41  6 39 16 33 30 51 


如上表，每个格子里都填上了「第几个访问」。请确认 **0 到 63 各出现一次**。

## 验算找到的路线

Z3 给出的答案是否真的构成骑士巡游？这里再写一段自己检查的代码。

In [3]:
def check_tour(board):
    """按骑士的移动顺序，逐个确认每个格子的值"""
    h, w = len(board), len(board[0])
    pos = {board[i][j]: (i, j) for i in range(h) for j in range(w)}

    if sorted(pos) != list(range(h * w)):
        return "编号没有做到 0〜63 各一次"

    for k in range(h * w):
        i, j = pos[k]
        i2, j2 = pos[(k + 1) % (h * w)]
        if (i2 - i, j2 - j) not in KNIGHT_MOVES:
            return f"{k} 号格到 {k + 1} 号格的移动不是骑士的走法"

    return "OK: 按骑士走法 0 → 1 → … → 63 → 0 走完一圈"


tour = [[m[cells[i, j]].as_long() for j in range(w)] for i in range(h)]
print(check_tour(tour))

OK: 按骑士走法 0 → 1 → … → 63 → 0 走完一圈


## 约束的含义

| 约束 | 含义 |
|---|---|
| `Implies(c < N - 1, PbEq(next_is, 1))` | 除最后一个外，周围必须有一个 `c+1` 的格子 |
| `Implies(c > 0, PbEq(prev_is, 1))` | 除第一个外，周围必须有一个 `c-1` 的格子 |
| `Implies(c == N - 1, PbEq(start_is, 1))` | 最后一个格子的周围必须有一个 `0` 的格子 |
| `Distinct(...)` | 0〜63 不重复 |

「周围有一个 `c+1` 的格子」和「周围有一个 `c-1` 的格子」，其实只写一边意思也一样（从 `c+1` 的格子来看，`c` 就是 `c-1`）。两边都写上，Z3 更容易搜索。

## 什么是 `PbEq()`

`PbEq(args, k)` 是**加权和恰好等于 `k`** 的约束。`args` 是 `(条件, 权重)` 的列表，条件为 `True` 时就把权重加进去。

例如：权重为 1 的条件有 2 个为真、权重为 2 的有 1 个为真，合计就是 4。

In [4]:
x, y, z = Bools('x y z')

s = Solver()
s.add(PbEq([(x, 1), (y, 2), (z, 3)], 5))
s.check()
print(s.model())

[y = True, z = True, x = False]


`x` 得 1 分、`y` 得 2 分、`z` 得 3 分，于是选出了合计恰好为 5 的组合（`y` 和 `z` 为真）。

骑士巡游的代码里，所有条件的权重都是 1。也就是说 `PbEq(..., 1)` 表示**「恰好有一个条件为真」**。

用 `Sum(...) == 1` 也能写出同样的意思，但 `PbEq()` 更能表达「恰好一个」的意图，Z3 解起来也更快。

## 小结

- 把移动顺序写成每个格子的整数，用「相邻格子是骑士的走法」这一约束就能定下路线
- 用 `Distinct` 保证 0〜63 各用一次
- 「恰好一个」可以用 `PbEq([...], 1)` 写出来
- 对得到的答案，准备一段自己验算的代码会更放心（`check_tour()`）